# Deep Learning Training on SageMaker
This notebook demonstrates how to run a deep learning training job on AWS SageMaker using your custom training script in `src/train.py`.

In [ ]:
# Install required dependencies (if running locally, uncomment below)
# !pip install accelerate ipykernel ipywidgets jupyter loguru pandas ruff scikit-learn torch transformers sagemaker

## Set up SageMaker environment and permissions

In [ ]:
import sagemaker
from sagemaker import get_execution_role
import boto3
import os

role = get_execution_role()  # If running in SageMaker notebook instance
session = sagemaker.Session()
bucket = "bucket"  # Replace with your S3 bucket name
s3_data_path = f"s3://{bucket}/data"

## Upload your training script and requirements to S3 (if needed)

In [ ]:
# If your training script is not already in S3, upload it
# session.upload_data(path='src', bucket=bucket, key_prefix='src')

## Define the SageMaker PyTorch Estimator

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="src/train.py",
    source_dir="src",
    role=role,
    framework_version="2.0",  # Match your torch version if needed
    py_version="py3",
    instance_count=1,
    instance_type="ml.p3.2xlarge",  # Change as needed
    hyperparameters={
        "data_path": s3_data_path,
        # Add other hyperparameters if needed
    },
    output_path=f"s3://{bucket}/output",
)

## Launch the training job

In [ ]:
estimator.fit({"train": s3_data_path})

## Monitor and retrieve training results

In [ ]:
# After training, you can access model artifacts in S3
print("Model artifacts saved to:", estimator.model_data)